In [1]:
import json
import csv
import glob
import os
import argparse
from collections import defaultdict

In [ ]:
def load_posts_from_files(file_path):
    """
    Reads a single .jsonl file and loads it into a dictionary.
    Args:
        file_path: Path to a single .jsonl file
    Returns:
        posts_map: dict {post_id: post_data_dict}
        children_map: dict {parent_post_id: [list_of_child_post_ids]}
    """
    posts_map = {}
    children_map = defaultdict(list)
    
    # Check if file exists
    if not os.path.isfile(file_path):
        print(f"File not found: {file_path}")
        return {}, {}
    
    if not file_path.endswith('.jsonl'):
        print(f"Warning: File {file_path} does not have .jsonl extension")

    print(f"Loading data from {file_path}...")
    
    with open(file_path, 'r', encoding='utf-8') as f:
        for line in f:
            try:
                # Clean up line and parse JSON
                line = line.strip()
                if not line:
                    continue
                post = json.loads(line)
                
                pid = post.get('post_id')
                if not pid:
                    continue
                    
                # Store post data (deduplicate by ID)
                if pid not in posts_map:
                    posts_map[pid] = post
                    
                    # Build adjacency relationship
                    # reply_to indicates the parent. 
                    # If reply_to is null, it might be a root.
                    parent_id = post.get('reply_to')
                    if parent_id:
                        children_map[parent_id].append(pid)
                        
            except json.JSONDecodeError:
                continue
                
    print(f"Total unique posts loaded: {len(posts_map)}")
    return posts_map, children_map



In [5]:
posts_map, children_map = load_posts_from_files('agent/data/bsky/user_posts/39699.jsonl')

Loading data from agent/data/bsky/user_posts/39699.jsonl...
Total unique posts loaded: 10151


In [8]:

def extract_conversations(posts_map, children_map):
    """
    Finds chains of length 3: OP -> Reply1 -> Reply2
    """
    threads_data = []
    
    # 1. Identify Potential Roots
    # A root is usually defined as having no reply_to, OR its thread_root equals its post_id
    # We iterate over all posts to find valid starting points for our chains
    
    print("reconstructing conversation threads...")
    
    # We track processed chains to avoid duplicates
    processed_chains = set()

    for pid, post in posts_map.items():
        # Check if this is a Root (OP)
        # Criteria: No reply_to, or explicitly marked as root
        reply_to = post.get('reply_to')
        thread_root = post.get('thread_root')
        
        is_root = (reply_to is None) or (thread_root == pid)
        
        if is_root:
            op_id = pid
            
            # LEVEL 1: Look for Author 1 (Direct replies to OP)
            replies_to_op = children_map.get(op_id, [])
            
            for reply1_id in replies_to_op:
                reply1 = posts_map[reply1_id]
                
                # LEVEL 2: Look for Author 2 (Replies to Author 1)
                replies_to_r1 = children_map.get(reply1_id, [])
                
                for reply2_id in replies_to_r1:
                    reply2 = posts_map[reply2_id]
                    
                    # We have found a chain: OP -> Reply1 -> Reply2
                    # Create a unique Thread ID for this specific interaction chain
                    # (We use the root ID + suffix to distinguish distinct branches if needed)
                    chain_id = f"{op_id}_{reply1_id}_{reply2_id}"
                    
                    if chain_id in processed_chains:
                        continue
                        
                    # Add Sequence 1: OP
                    threads_data.append({
                        "thread_id": chain_id,
                        "sequence": 1,
                        "author_id": post.get('user_id'),
                        "content": post.get('text', "")
                    })
                    
                    # Add Sequence 2: Author 1
                    threads_data.append({
                        "thread_id": chain_id,
                        "sequence": 2,
                        "author_id": reply1.get('user_id'),
                        "content": reply1.get('text', "")
                    })
                    
                    # Add Sequence 3: Author 2
                    threads_data.append({
                        "thread_id": chain_id,
                        "sequence": 3,
                        "author_id": reply2.get('user_id'),
                        "content": reply2.get('text', "")
                    })
                    
                    # Add Sequence 4: OP (Closing the loop as per your diagram)
                    # "OP | Author_1 | Author_2 | OP"
                    # We check if the OP replied back to Author 2
                    replies_to_r2 = children_map.get(reply2_id, [])
                    op_closing_reply = None
                    
                    for reply3_id in replies_to_r2:
                        reply3 = posts_map[reply3_id]
                        if reply3.get('user_id') == post.get('user_id'):
                            op_closing_reply = reply3
                            break # Found the OP coming back
                    
                    if op_closing_reply:
                        threads_data.append({
                            "thread_id": chain_id,
                            "sequence": 4,
                            "author_id": op_closing_reply.get('user_id'),
                            "content": op_closing_reply.get('text', "")
                        })
                    
                    processed_chains.add(chain_id)

    return threads_data

In [10]:
data = extract_conversations(posts_map, children_map)

reconstructing conversation threads...


In [11]:
data

[{'thread_id': '139745114_139745113_139745112',
  'sequence': 1,
  'author_id': 39699,
  'content': 'There’s a kind of person who has internalized that having to take personal responsibility for anything is foolish under capitalism, so they say things like “it’s society’s fault I have to drink and drive”. It should make you realize you, as an adult, do need to take responsibility sometimes.'},
 {'thread_id': '139745114_139745113_139745112',
  'sequence': 2,
  'author_id': 39699,
  'content': 'If you think this post is bad you should just be happy it’s not a side by side with the dril “it’s impossible to say if drink driving is bad or not” tweet'},
 {'thread_id': '139745114_139745113_139745112',
  'sequence': 3,
  'author_id': 39699,
  'content': 'Anyway I think the more useful takeaway here is that there are comedians who took the big moral stances correctly and are charismatic that you absolutely should not take advice from on any topic'},
 {'thread_id': '8665808_139745150_139745149',

In [ ]:
keys = ["thread_id", "sequence", "author_id", "content"]
with open(output_file, 'w', newline='', encoding='utf-8') as f:
    writer = csv.DictWriter(f, fieldnames=keys)
    writer.writeheader()
    writer.writerows(data)
print(f"Successfully saved to {args.output_file}")